In [1]:
import pandas as pd
import numpy as np
from thefuzz import process, fuzz

In [2]:
# %pip install thefuzz

In [3]:
v1 = pd.read_csv('clear_data/df_prices_v1.csv')
v1

,price,mini,faction
0,70.84,Custodian Dreadnought,Adeptus_Custodes
1,67.62,Sentinel Guard Sodality,Adeptus_Custodes
2,40.02,Shield Captain,Adeptus_Custodes
3,75.44,Venatari Sodality,Adeptus_Custodes
4,67.62,Custodian Guard Sodality,Adeptus_Custodes
...,...,...,...
835,119.60,C'tan Shard of the Nightbringer,Necrons
836,59.80,Victrix Honour Guard,Supplement_Marines
837,51.06,Marneus Calgar in Armour of Antilochus,Supplement_Marines
838,178.48,Knight Preceptor/Canis Rex,Imperial_Knights


In [4]:
v2 = pd.read_csv('clear_data/df_prices_v2.csv')
v2

,price,mini,faction
0,56.12,Custodian Guard,Adeptus_Custodes
1,46.00,Sisters Of Silence,Adeptus_Custodes
2,35.88,Captain General Trajann Valoris,Adeptus_Custodes
3,56.12,Custodian Wardens,Adeptus_Custodes
4,56.12,Vertus Praetors,Adeptus_Custodes
...,...,...,...
380,28.52,Bloodmaster Herald Of Khorne,Chaos_Daemons
381,46.92,Daemons Of Khorne Flesh Hounds,Chaos_Daemons
382,34.04,Karanak The Hound Of Vengeance,Chaos_Daemons
383,32.20,Daemons Of Slaanesh The Masque,Chaos_Daemons


In [5]:
mat = pd.read_csv('clear_data/df_materials.csv')
mat

,mini,faction,material,edition,year
0,Baharroth Wings,Craftworlds,Metal,Warhammer 40K: Rogue Trader,1990.0
1,Warp Spiders,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0
2,Avatar of Khaine,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0
3,Asurmen,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0
4,Karandras,Craftworlds,Finecast,Warhammer 40K: 2nd Edition,1994.0
...,...,...,...,...,...
977,Ahriman,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2016.0
978,Tzaangor Enlightened,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2017.0
979,Tzaangor Skyfires,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2017.0
980,Tzaangor Shaman,Thousand_Sons,Plastic,Warhammer 40K: 7th Edition,2017.0


In [6]:
mod = pd.read_csv('clear_data/df_models.csv')
mod

,mini,base_mm2,cost
0,Warboss,40mm,75.0
1,Warboss In Mega Armour,50mm,80.0
2,Warboss On Warbike,100 x 40mm,75.0
3,Weirdboy,40mm,65.0
4,Big Mek In Mega Armour,40mm,90.0
...,...,...,...
2208,Defiler,160mm,NaN
2209,Thulia Ghuld,80mm,NaN
2210,Hastarii Exterminators,32mm,NaN
2211,Hastarii Fusiliers,32mm,NaN


In [7]:
alfa = mod.merge(
    mat[['mini', 'faction']],
    on='mini',
    how='left'
)
alfa[alfa['faction'].isna()]

,mini,base_mm2,cost,faction
0,Warboss,40mm,75.0,NaN
1,Warboss In Mega Armour,50mm,80.0,NaN
2,Warboss On Warbike,100 x 40mm,75.0,NaN
4,Big Mek In Mega Armour,40mm,90.0,NaN
5,Big Mek On Warbike,90 x 52mm,105.0,NaN
...,...,...,...,...
2322,Ferren Areios,40mm,NaN,NaN
2330,Thulia Ghuld,80mm,NaN,NaN
2331,Hastarii Exterminators,32mm,NaN,NaN
2332,Hastarii Fusiliers,32mm,NaN,NaN


In [8]:
faction_map = pd.concat([v1[['mini', 'faction']], v2[['mini', 'faction']]]).drop_duplicates('mini')
faction_map = faction_map.set_index('mini')['faction']

mask = alfa['faction'].isna()
alfa.loc[mask, 'faction'] = alfa.loc[mask, 'mini'].map(faction_map)

display(alfa[alfa['faction'].isna()])
alfa

,mini,base_mm2,cost,faction
0,Warboss,40mm,75.0,NaN
1,Warboss In Mega Armour,50mm,80.0,NaN
2,Warboss On Warbike,100 x 40mm,75.0,NaN
4,Big Mek In Mega Armour,40mm,90.0,NaN
5,Big Mek On Warbike,90 x 52mm,105.0,NaN
...,...,...,...,...
2320,Corsair Skyreavers,28.5mm,NaN,NaN
2321,Starfangs,105 x 70mm,NaN,NaN
2322,Ferren Areios,40mm,NaN,NaN
2331,Hastarii Exterminators,32mm,NaN,NaN


,mini,base_mm2,cost,faction
0,Warboss,40mm,75.0,NaN
1,Warboss In Mega Armour,50mm,80.0,NaN
2,Warboss On Warbike,100 x 40mm,75.0,NaN
3,Weirdboy,40mm,65.0,Orks
4,Big Mek In Mega Armour,40mm,90.0,NaN
...,...,...,...,...
2329,Defiler,160mm,NaN,Death_Guard
2330,Thulia Ghuld,80mm,NaN,Adeptus_Mechanicus
2331,Hastarii Exterminators,32mm,NaN,NaN
2332,Hastarii Fusiliers,32mm,NaN,NaN


In [9]:
# Crear un diccionario de nombres canonical
nombres_canonicos = mat['mini'].unique()

def match_mini(nombre, opciones, threshold=85):
    match, score = process.extractOne(nombre, opciones, scorer=fuzz.partial_ratio)
    return match if score >= threshold else nombre

# Aplicar a cada DF
v1['mini_matched'] = v1['mini'].apply(lambda x: match_mini(x, nombres_canonicos))
v2['mini_matched'] = v2['mini'].apply(lambda x: match_mini(x, nombres_canonicos))
mod['mini_matched'] = mod['mini'].apply(lambda x: match_mini(x, nombres_canonicos))

In [10]:
alfa[alfa['faction'].isna()]

,mini,base_mm2,cost,faction
0,Warboss,40mm,75.0,NaN
1,Warboss In Mega Armour,50mm,80.0,NaN
2,Warboss On Warbike,100 x 40mm,75.0,NaN
4,Big Mek In Mega Armour,40mm,90.0,NaN
5,Big Mek On Warbike,90 x 52mm,105.0,NaN
...,...,...,...,...
2320,Corsair Skyreavers,28.5mm,NaN,NaN
2321,Starfangs,105 x 70mm,NaN,NaN
2322,Ferren Areios,40mm,NaN,NaN
2331,Hastarii Exterminators,32mm,NaN,NaN


In [11]:
faction_map = {
    'Craftworld': 'Craftworlds',
    'Aeldari': 'Craftworlds',
    'Drukhari': 'Drukhari',
    'Dark Eldar': 'Drukhari',
    'Harlequin': 'Harlequins',
    'Ynnari': 'Ynnari',
    'Genestealer': 'Genestealer_Cult',
    'GSC': 'Genestealer_Cult',
    'Tyranids': 'Tyranids',
    'Tyranid': 'Tyranids',
    'Necron': 'Necrons',
    'Ork': 'Orks',
    'Tau': 'Tau_Empire',
    "T'au": 'Tau_Empire',
    "T’au": 'Tau_Empire',
    'Space Marine': 'Space_Marines',
    'Space Marines': 'Space_Marines',
    'Primaris': 'Space_Marines',
    'Blood Angel': 'Blood_Angels',
    'Dark Angel': 'Dark_Angels',
    'Space Wolf': 'Space_Wolves',
    'Deathwatch': 'Deathwatch',
    'Grey Knight': 'Grey_Knights',
    'Astra Militarum': 'Astra_Militarum',
    'Sororitas': 'Adepta_Sororitas',
    'Sister': 'Adepta_Sororitas',
    'Adepta': 'Adepta_Sororitas',
    'Custodes': 'Adeptus_Custodes',
    'Adeptus Mechanicus': 'Adeptus_Mechanicus',
    'AdMech': 'Adeptus_Mechanicus',
    'Imperial Assassin': 'Imperial_Assassins',
    'Assassin': 'Imperial_Assassins',
    'Imperial Knight': 'Imperial_Knights',
    'Inquisitor': 'Inquisition',
    'Inquisition': 'Inquisition',
    'Chaos Daemon': 'Chaos_Daemons',
    'Daemon': 'Chaos_Daemons',
    'Chaos Knight': 'Chaos_Knights',
    'Chaos Space Marine': 'Chaos_Space_Marines',
    'CSM': 'Chaos_Space_Marines',
    'Death Guard': 'Death_Guard',
    'Thousand Son': 'Thousand_Sons',
    # Nuevos del segundo listado:
    'Astra Telepathica': 'Adeptus_Custodes',
    'Tallyman': 'Chaos_Daemons',
    'Templars': 'Supplement_Marines',
    'Angels': 'Supplement_Marines',
    'Black Legion': 'Chaos_Space_Marines',
    'Ultramarines': 'Supplement_Marines',
    'Iron Hand': 'Supplement_Marines',
    'Raven Guard': 'Supplement_Marines',
    'Salamanders': 'Supplement_Marines',
    'Imperial Fist': 'Supplement_Marines',
    'White Scar': 'Supplement_Marines',
    'S/Marine': 'Space_Marines',
    'Steel Legion': 'Astra_Militarum',
    'A/I': 'Astra_Militarum',
    'Eldar Dark': 'Drukhari',
    'Gunship': 'Tau_Empire',
    'Cult': 'Chaos_Daemons',
    'Genestealers': 'Genestealer_Cult',
    'Assassinorum': 'Imperial_Assassins',
    'A/Soror': 'Adepta_Sororitas',
}

name_map = {
    'Farseer': 'Craftworlds',
    'Howling Banshee': 'Craftworlds',
    'Custodian': 'Adeptus_Custodes',
    'Dire Avenger': 'Craftworlds',
    'Sodality': 'Adeptus_Custodes',
    'Warp Spider': 'Craftworlds',
    'Swooping Hawk': 'Craftworlds',
    'Striking Scorpion': 'Craftworlds',
    'Fire Dragon': 'Craftworlds',
    'Guardian': 'Craftworlds',
    'Warlock': 'Craftworlds',
    'Avatar': 'Craftworlds',
    'Vyper': 'Craftworlds',
    'Wave Serpent': 'Craftworlds',
    'Wraithguard': 'Craftworlds',
    'Wraithlord': 'Craftworlds',
    'Wraithknight': 'Craftworlds',
    'Ranger': 'Craftworlds',
    'Shining Spear': 'Craftworlds',
    'Kabalite': 'Drukhari',
    'Wych': 'Drukhari',
    'Incubi': 'Drukhari',
    'Haemonculus': 'Drukhari',
    'Archon': 'Drukhari',
    'Succubus': 'Drukhari',
    'Reaver': 'Drukhari',
    'Hellion': 'Drukhari',
    'Mandrake': 'Drukhari',
    'Ravager': 'Drukhari',
    'Raider': 'Drukhari',
    'Venom': 'Drukhari',
    'Cronos': 'Drukhari',
    'Talos': 'Drukhari',
    'Lelith': 'Drukhari',
    'Drazhar': 'Drukhari',
    'Grotesque': 'Drukhari',
    'Wrack': 'Drukhari',
    'Scourge': 'Drukhari',
    'Troupe': 'Harlequins',
    'Shadowseer': 'Harlequins',
    'Death Jester': 'Harlequins',
    'Solitaire': 'Harlequins',
    'Voidweaver': 'Harlequins',
    'Starweaver': 'Harlequins',
    'Yvraine': 'Ynnari',
    'The Visarch': 'Ynnari',
    'The Yncarne': 'Ynnari',
    'Cultist': 'Genestealer_Cult',
    'Acolyte': 'Genestealer_Cult',
    'Patriarch': 'Genestealer_Cult',
    'Magus': 'Genestealer_Cult',
    'Metamorph': 'Genestealer_Cult',
    'Neophyte': 'Genestealer_Cult',
    'Aberrant': 'Genestealer_Cult',
    'Kelermorph': 'Genestealer_Cult',
    'Hive Guard': 'Tyranids',
    'Hormagaunt': 'Tyranids',
    'Termagant': 'Tyranids',
    'Gaunt': 'Tyranids',
    'Warrior': 'Tyranids',
    'Carnifex': 'Tyranids',
    'Trygon': 'Tyranids',
    'Ravener': 'Tyranids',
    'Lictor': 'Tyranids',
    'Haruspex': 'Tyranids',
    'Zoanthrope': 'Tyranids',
    'Neurotyrant': 'Tyranids',
    'Biovores': 'Tyranids',
    'Myphitic': 'Tyranids',
    'Tervigon': 'Tyranids',
    'Old One Eye': 'Tyranids',
    'The Swarmlord': 'Tyranids',
    'Hive Tyrant': 'Tyranids',
    'Neurothrope': 'Tyranids',
    'Immortal': 'Necrons',
    'Lychguard': 'Necrons',
    'Deathmark': 'Necrons',
    'Flayed One': 'Necrons',
    'Canoptek': 'Necrons',
    'Monolith': 'Necrons',
    'Ghost Ark': 'Necrons',
    'Night Scythe': 'Necrons',
    'Doom Scythe': 'Necrons',
    'Tomb Blade': 'Necrons',
    'Spyder': 'Necrons',
    'Overlord': 'Necrons',
    'Cryptek': 'Necrons',
    'Lord': 'Necrons',
    'Triarch': 'Necrons',
    'Wraith': 'Necrons',
    'Destroyer': 'Necrons',
    'Cairn': 'Necrons',
    'Boyz': 'Orks',
    'Nob': 'Orks',
    'Warboss': 'Orks',
    'Mek': 'Orks',
    'Weirdboy': 'Orks',
    'Dok': 'Orks',
    'Flash Gitz': 'Orks',
    'Burna': 'Orks',
    'Shoota': 'Orks',
    'Slugga': 'Orks',
    'Trukk': 'Orks',
    'Deff Dread': 'Orks',
    'Killa Kan': 'Orks',
    'Looted Wagon': 'Orks',
    'Big Mek': 'Orks',
    'Ghazghkull': 'Orks',
    'Gretchin': 'Orks',
    'Snotling': 'Orks',
    'Fire Warrior': 'Tau_Empire',
    'Crisis': 'Tau_Empire',
    'Broadside': 'Tau_Empire',
    'Riptide': 'Tau_Empire',
    'Pathfinder': 'Tau_Empire',
    'Stealth': 'Tau_Empire',
    'Piranha': 'Tau_Empire',
    'Hammerhead': 'Tau_Empire',
    'Skyray': 'Tau_Empire',
    'XV': 'Tau_Empire',
    'Commander': 'Tau_Empire',
    'Ethereal': 'Tau_Empire',
    'Devilfish': 'Tau_Empire',
    'Intercessor': 'Space_Marines',
    'Tactical': 'Space_Marines',
    'Devastator': 'Space_Marines',
    'Assault': 'Space_Marines',
    'Sternguard': 'Space_Marines',
    'Vanguard': 'Space_Marines',
    'Scout': 'Space_Marines',
    'Aggressor': 'Space_Marines',
    'Eliminator': 'Space_Marines',
    'Inceptor': 'Space_Marines',
    'Suppressor': 'Space_Marines',
    'Reiver': 'Space_Marines',
    'Hellblaster': 'Space_Marines',
    'Outrider': 'Space_Marines',
    'Whirlwind': 'Space_Marines',
    'Rhino': 'Space_Marines',
    'Predator': 'Space_Marines',
    'Land Raider': 'Space_Marines',
    'Republik': 'Space_Marines',
    'Impulsor': 'Space_Marines',
    'Gladiator': 'Space_Marines',
    'Sanguinius': 'Blood_Angels',
    'Dante': 'Blood_Angels',
    'Death Company': 'Blood_Angels',
    'Furioso': 'Blood_Angels',
    'Baal': 'Blood_Angels',
    'Lemartes': 'Blood_Angels',
    'Inner Circle': 'Dark_Angels',
    'Belial': 'Dark_Angels',
    'Ravenwing': 'Dark_Angels',
    'Deathwing': 'Dark_Angels',
    'Wolf': 'Space_Wolves',
    'Thunderwolf': 'Space_Wolves',
    'Rune Priest': 'Space_Wolves',
    'Iron Priest': 'Space_Wolves',
    'Wolf Lord': 'Space_Wolves',
    'Wulfen': 'Space_Wolves',
    'Blood Claw': 'Space_Wolves',
    'Stormwolf': 'Space_Wolves',
    'Kill Team': 'Deathwatch',
    'Watch Master': 'Deathwatch',
    'Veteran': 'Deathwatch',
    'Brother': 'Grey_Knights',
    'Justicar': 'Grey_Knights',
    'Castellan': 'Grey_Knights',
    'Venerable': 'Grey_Knights',
    'Nemesis': 'Grey_Knights',
    'Purifier': 'Grey_Knights',
    'Cadian': 'Astra_Militarum',
    'Krieg': 'Astra_Militarum',
    'Kasrkin': 'Astra_Militarum',
    'Conscript': 'Astra_Militarum',
    'Chimera': 'Astra_Militarum',
    'Leman Russ': 'Astra_Militarum',
    'Basilisk': 'Astra_Militarum',
    'Baneblade': 'Astra_Militarum',
    'Trooper': 'Astra_Militarum',
    'Ogryn': 'Astra_Militarum',
    'Ratling': 'Astra_Militarum',
    'Commissar': 'Astra_Militarum',
    'Lord Solar': 'Astra_Militarum',
    'Tempestus': 'Astra_Militarum',
    'Canoness': 'Adepta_Sororitas',
    'Palatine': 'Adepta_Sororitas',
    'Seraphim': 'Adepta_Sororitas',
    'Celestian': 'Adepta_Sororitas',
    'Dominion': 'Adepta_Sororitas',
    'Battle Sister': 'Adepta_Sororitas',
    'Novitiate': 'Adepta_Sororitas',
    'Hospitaller': 'Adepta_Sororitas',
    'Preacher': 'Adepta_Sororitas',
    'Imagifier': 'Adepta_Sororitas',
    'Celestine': 'Adepta_Sororitas',
    'Paragon': 'Adepta_Sororitas',
    'Exorcist': 'Adepta_Sororitas',
    'Retributor': 'Adepta_Sororitas',
    'Shield Captain': 'Adeptus_Custodes',
    'Vertus Praetor': 'Adeptus_Custodes',
    'Sagittarum': 'Adeptus_Custodes',
    'Vexilus': 'Adeptus_Custodes',
    'Valiant': 'Adeptus_Custodes',
    'Telemon': 'Adeptus_Custodes',
    'Caladius': 'Adeptus_Custodes',
    'Coronus': 'Adeptus_Custodes',
    'Contemptor': 'Adeptus_Custodes',
    'Allarus': 'Adeptus_Custodes',
    'Trajann': 'Adeptus_Custodes',
    'Solaria': 'Adeptus_Custodes',
    'Skorpius': 'Adeptus_Mechanicus',
    'Kastelan': 'Adeptus_Mechanicus',
    'Sydonian': 'Adeptus_Mechanicus',
    'Ironstrider': 'Adeptus_Mechanicus',
    'Techpriest': 'Adeptus_Mechanicus',
    'Enginseer': 'Adeptus_Mechanicus',
    'Secutarii': 'Adeptus_Mechanicus',
    'Peltast': 'Adeptus_Mechanicus',
    'Hoplite': 'Adeptus_Mechanicus',
    'Hastarii': 'Adeptus_Mechanicus',
    'Maniple': 'Adeptus_Mechanicus',
    'Kataphron': 'Adeptus_Mechanicus',
    'Onslaught': 'Adeptus_Mechanicus',
    'Cawl': 'Adeptus_Mechanicus',
    'Vindicare': 'Imperial_Assassins',
    'Eversor': 'Imperial_Assassins',
    'Callidus': 'Imperial_Assassins',
    'Culexus': 'Imperial_Assassins',
    'Paladin': 'Imperial_Knights',
    'Errant': 'Imperial_Knights',
    'Crusader': 'Imperial_Knights',
    'Warden': 'Imperial_Knights',
    'Gallant': 'Imperial_Knights',
    'Armiger': 'Imperial_Knights',
    'Moira': 'Imperial_Knights',
    'Questoris': 'Imperial_Knights',
    'Warmaster': 'Imperial_Knights',
    'Bloodthirster': 'Chaos_Daemons',
    'Lord of Change': 'Chaos_Daemons',
    'Great Unclean One': 'Chaos_Daemons',
    'Keeper of Secrets': 'Chaos_Daemons',
    'Bloodletter': 'Chaos_Daemons',
    'Horror': 'Chaos_Daemons',
    'Plaguebearer': 'Chaos_Daemons',
    'Daemonette': 'Chaos_Daemons',
    'Screamer': 'Chaos_Daemons',
    'Exalted': 'Chaos_Daemons',
    'Desecrator': 'Chaos_Knights',
    'Decimator': 'Chaos_Knights',
    'Shroud': 'Chaos_Knights',
    'Stalker': 'Chaos_Knights',
    'Warg': 'Chaos_Knights',
    'Abominant': 'Chaos_Knights',
    'Mourneblade': 'Chaos_Knights',
    'Legion': 'Chaos_Space_Marines',
    'Champion': 'Chaos_Space_Marines',
    'Obliterator': 'Chaos_Space_Marines',
    'Possessed': 'Chaos_Space_Marines',
    'Havoc': 'Chaos_Space_Marines',
    'Raptor': 'Chaos_Space_Marines',
    'Sorcerer': 'Chaos_Space_Marines',
    'Terminator': 'Chaos_Space_Marines',
    'Mutilators': 'Chaos_Space_Marines',
    'Plague': 'Death_Guard',
    'Nurgle': 'Death_Guard',
    'Mortarion': 'Death_Guard',
    'Poxwalker': 'Death_Guard',
    'Poxbrute': 'Death_Guard',
    'Foetid Bloat': 'Death_Guard',
    'Blasphemy': 'Death_Guard',
    'Spawn': 'Death_Guard',
    'Tzeentch': 'Thousand_Sons',
    'Magnus': 'Thousand_Sons',
    'Rubric': 'Thousand_Sons',
    'Chaos Lord': 'Thousand_Sons',
    'Mutalith': 'Thousand_Sons',
    'Thulia': 'Adeptus_Mechanicus',
    'Daemon': 'Chaos_Daemons',
    'Khorne': 'Chaos_Daemons',
    'Kharn': 'Chaos_Daemons',
    'Skarbrand': 'Chaos_Daemons',
    'Syll': 'Chaos_Daemons',
    'Feculent': 'Chaos_Daemons',
    'Karanak': 'Chaos_Daemons',
    'Eisenhorn': 'Inquisition',
    'Ravenor': 'Inquisition',
    'Lord Inquisitor': 'Inquisition',
    'Lord Commissar': 'Inquisition',
    'Ciaphas Cain': 'Inquisition',
    'Marneus': 'Supplement_Marines',
    'Militarum': 'Astra_Militarum',
    'Aeronautica': 'Astra_Militarum',
    'Aer': 'Astra_Militarum',
    'Captain General': 'Adeptus_Custodes',
    'Daemons': 'Chaos_Daemons',
    'Chaos': 'Chaos_Daemons',
    'Saint': 'Adepta_Sororitas',
    'Knight': 'Imperial_Knights',
    'Titan': 'Imperial_Knights',
    'Mech': 'Adeptus_Mechanicus',
    'Talons': 'Adeptus_Custodes',
    'Tufts': 'Space_Marines',
    'Space Hulk': 'Space_Marines',
    'reavers': 'Chaos_Space_Marines',
    "C'tan" : 'Necrons',
    "C’tan" : 'Necrons',
    'Victrix': 'Supplement_Marines',
    "Emperor's Children": 'Chaos_Space_Marines',
    "World Eaters": 'Chaos_Space_Marines',
    "Imperial Agents": 'Deathwatch',
    'Seraptek': 'Necrons',
    'Leagues of Votann': 'Leagues_of_Votann',
    'Dorn': 'Supplement_Marines'
}

In [12]:
# 1. Normalizar faction existente con faction_map
alfa['faction'] = alfa['faction'].replace(faction_map)

# 2. Inferir faction desde name_map
def infer_faction(mini_name, mapping):
    mini_lower = str(mini_name).lower()
    for key, faction in mapping.items():
        if key.lower() in mini_lower:
            return faction
    return np.nan

mask = alfa['faction'].isna()
alfa.loc[mask, 'faction'] = alfa.loc[mask, 'mini'].apply(lambda x: infer_faction(x, name_map))

print(f"Sin faction: {alfa['faction'].isna().sum()}")
display(alfa[alfa['faction'].isna()])
alfa

Sin faction: 785


,mini,base_mm2,cost,faction
7,MAKARI,25mm,235.0,NaN
13,Painboy On Warbike,90 x 52mm,85.0,NaN
15,BOY,32mm,80.0,NaN
16,BOY,32mm,170.0,NaN
21,RUNTHERD,32mm,40.0,NaN
...,...,...,...,...
2312,"GARLON SOULEATER, GARREON THE CORPSEMASTER, KA...",40mm,NaN,NaN
2313,"CAPTAIN SARGOTTA, THE ENFORCER",40mm,NaN,NaN
2316,Starfangs,105 x 70mm,NaN,NaN
2321,Starfangs,105 x 70mm,NaN,NaN


,mini,base_mm2,cost,faction
0,Warboss,40mm,75.0,Orks
1,Warboss In Mega Armour,50mm,80.0,Orks
2,Warboss On Warbike,100 x 40mm,75.0,Orks
3,Weirdboy,40mm,65.0,Orks
4,Big Mek In Mega Armour,40mm,90.0,Orks
...,...,...,...,...
2329,Defiler,160mm,NaN,Death_Guard
2330,Thulia Ghuld,80mm,NaN,Adeptus_Mechanicus
2331,Hastarii Exterminators,32mm,NaN,Adeptus_Mechanicus
2332,Hastarii Fusiliers,32mm,NaN,Adeptus_Mechanicus


In [13]:
from difflib import SequenceMatcher

# Crear referencia de mini -> faction con las que ya tienen datos
known = alfa[alfa['faction'].notna()][['mini', 'faction']].copy()

def find_best_match(mini_name, known_df, threshold=0.75):
    best_score = 0
    best_faction = np.nan
    mini_lower = str(mini_name).lower()
    
    for _, row in known_df.iterrows():
        score = SequenceMatcher(None, mini_lower, str(row['mini']).lower()).ratio()
        if score > best_score:
            best_score = score
            best_faction = row['faction']
    
    return best_faction if best_score >= threshold else np.nan

mask = alfa['faction'].isna()
alfa.loc[mask, 'faction'] = alfa.loc[mask, 'mini'].apply(
    lambda x: find_best_match(x, known)
)

print(f"Sin faction: {alfa['faction'].isna().sum()}")
display(alfa[alfa['faction'].isna()].head(50))
alfa

Sin faction: 704


,mini,base_mm2,cost,faction
7,MAKARI,25mm,235.0,NaN
15,BOY,32mm,80.0,NaN
16,BOY,32mm,170.0,NaN
21,RUNTHERD,32mm,40.0,NaN
22,RUNTHERD,32mm,80.0,NaN
25,TANKBUSTAS,32mm,140.0,NaN
34,KOMMANDOS,32mm,120.0,NaN
37,STORMBOY,32mm,65.0,NaN
38,STORMBOY,32mm,130.0,NaN
47,WARBIKER,75 x 42mm,65.0,NaN


,mini,base_mm2,cost,faction
0,Warboss,40mm,75.0,Orks
1,Warboss In Mega Armour,50mm,80.0,Orks
2,Warboss On Warbike,100 x 40mm,75.0,Orks
3,Weirdboy,40mm,65.0,Orks
4,Big Mek In Mega Armour,40mm,90.0,Orks
...,...,...,...,...
2329,Defiler,160mm,NaN,Death_Guard
2330,Thulia Ghuld,80mm,NaN,Adeptus_Mechanicus
2331,Hastarii Exterminators,32mm,NaN,Adeptus_Mechanicus
2332,Hastarii Fusiliers,32mm,NaN,Adeptus_Mechanicus


In [14]:
from difflib import SequenceMatcher

# 1. Re-run fuzzy match con threshold más bajo (0.6)
known = alfa[alfa['faction'].notna()][['mini', 'faction']].drop_duplicates('mini')

def find_best_match(mini_name, known_df, threshold=0.6):
    best_score = 0
    best_faction = np.nan
    mini_lower = str(mini_name).lower()
    for _, row in known_df.iterrows():
        score = SequenceMatcher(None, mini_lower, str(row['mini']).lower()).ratio()
        if score > best_score:
            best_score = score
            best_faction = row['faction']
    return best_faction if best_score >= threshold else np.nan

mask = alfa['faction'].isna()
alfa.loc[mask, 'faction'] = alfa.loc[mask, 'mini'].apply(
    lambda x: find_best_match(x, known)
)

# 3. Buscar por contención en lista de known
def match_by_containment(mini_name, known_df):
    mini_words = set(str(mini_name).lower().split())
    for _, row in known_df.iterrows():
        known_words = set(str(row['mini']).lower().split())
        common = mini_words & known_words
        if len(common) >= 2 and len(common) / len(mini_words) >= 0.5:
            return row['faction']
    return np.nan

mask = alfa['faction'].isna()
alfa.loc[mask, 'faction'] = alfa.loc[mask, 'mini'].apply(
    lambda x: match_by_containment(x, known)
)

print(f"Sin faction: {alfa['faction'].isna().sum()}")
display(alfa[alfa['faction'].isna()])
alfa

Sin faction: 301


,mini,base_mm2,cost,faction
25,TANKBUSTAS,32mm,140.0,NaN
53,Skorchas,Use model,45.0,NaN
54,Skorchas,Use model,90.0,NaN
55,Warbuggies,Use model,40.0,NaN
56,Warbuggies,Use model,80.0,NaN
...,...,...,...,...
2301,Da Red Gobbo Tinboy,40mm,NaN,NaN
2305,Cato Sicarius,40mm,NaN,NaN
2307,Nekrosor Ammentar,80mm,NaN,NaN
2310,"Gaius Silva, Aemelia Minervas, Dainal Korneliu...",28.5mm,NaN,NaN


,mini,base_mm2,cost,faction
0,Warboss,40mm,75.0,Orks
1,Warboss In Mega Armour,50mm,80.0,Orks
2,Warboss On Warbike,100 x 40mm,75.0,Orks
3,Weirdboy,40mm,65.0,Orks
4,Big Mek In Mega Armour,40mm,90.0,Orks
...,...,...,...,...
2329,Defiler,160mm,NaN,Death_Guard
2330,Thulia Ghuld,80mm,NaN,Adeptus_Mechanicus
2331,Hastarii Exterminators,32mm,NaN,Adeptus_Mechanicus
2332,Hastarii Fusiliers,32mm,NaN,Adeptus_Mechanicus


In [15]:
character_map = {
    # ORKS
    'GHAZGHKULL THRAKA': 'Orks', 'MAKARI': 'Orks', 'Painboy On Warbike': 'Orks',
    'BOY': 'Orks', 'BOSS NOB': 'Orks', 'GRETCHIN': 'Orks', 'RUNTHERD': 'Orks',
    'TANKBUSTAS': 'Orks', 'KOMMANDOS': 'Orks', 'STORMBOY': 'Orks',
    'WARBIKER': 'Orks', 'BOSS NOB ON WARBIKE': 'Orks', 'Wartrakks': 'Orks',
    'Skorchas': 'Orks', 'Warbuggies': 'Orks', 'Big Gunz': 'Orks',
    'Grot Tanks': 'Orks', 'Grot Mega-tank': 'Orks', 'Squiggoth': 'Orks',
    'Meka-dread': 'Orks', 'Lifta Wagon': 'Orks', 'Big Trakk': 'Orks',
    'Kannonwagon': 'Orks', 'Kill Tank': 'Orks', 'Chinork Warkopta': 'Orks',
    'Gargantuan Squiggoth': 'Orks', 'Nob With Waaagh! Banner': 'Orks',
    'Nobz On Warbikes': 'Orks', 'Big Mek With Shokk Attack Gun': 'Orks',
    'Kustom Boosta-blasta': 'Orks', 'Da Red Gobbo': 'Orks', 'Da Red Gobbo Tinboy': 'Orks',
    'Carnodon': 'Orks', 'Deffkoptas With Big Shootas': 'Orks',
    'Mega Dread': 'Orks', 'Zodgrod Wortsnagga': 'Orks', 'Beastboss On Squigosaur': 'Orks',
    'Wurrboy': 'Orks', 'BEAST SNAGGA BOY': 'Orks', 'BEAST SNAGGA NOB': 'Orks',
    'Squighog Boyz': 'Orks', 'Nob On Smasha Squig': 'Orks', "Big'ed Bossbunka": 'Orks',
    'BREAKA BOY': 'Orks', 'Meka-dread': 'Orks',
    'WOLF SCOUTS': 'Space_Wolves', 'HUNTING WOLVES': 'Space_Wolves',
    
    # SPACE MARINES / SUPPLEMENT
    'Apothecary On Bike': 'Space_Marines', 'Assault Squad With Jump Packs': 'Space_Marines',
    'Land Raider Excelsior': 'Space_Marines', 'Suppressor Squad': 'Space_Marines',
    'Bladeguard Veteran Squad': 'Space_Marines', 'Caestus Assault Ram': 'Space_Marines',
    'Captain': 'Space_Marines', 'Lieutenant With Combi-weapon': 'Space_Marines',
    'Librarian In Terminator Armour': 'Space_Marines', 'Captain With Jump Pack': 'Space_Marines',
    'Rhino Primaris': 'Space_Marines', 'Firestrike Servo-turrets': 'Space_Marines',
    'Sicaran Venator': 'Space_Marines', 'Typhon': 'Space_Marines',
    'Ultramarines Honour Guard': 'Space_Marines', 'Carab Culln The Risen': 'Space_Marines',
    'Tarantula Sentry Battery': 'Space_Marines', 'Tarantula Air Defence Battery': 'Space_Marines',
    'Eradicator Squad': 'Space_Marines', 'Relic Terminator Squad': 'Space_Marines',
    'Javelin Attack Speeder': 'Space_Marines', 'Rapier Carrier': 'Space_Marines',
    'Mortis Dreadnought': 'Space_Marines', 'Chaplain With Jump Pack': 'Space_Marines',
    'Storm Eagle Gunship': 'Space_Marines', 'Chaplain In Terminator Armour': 'Space_Marines',
    'SPACE MARINE BIKE': 'Space_Marines', 'ATTACK BIKE': 'Space_Marines',
    'Terminator Assault Squad': 'Space_Marines', 'Librarian In Phobos Armour': 'Space_Marines',
    'Uriel Ventris': 'Space_Marines', 'Dreadnought Drop Pod': 'Space_Marines',
    'Sicaran Punisher': 'Space_Marines', 'Iron Father Feirros': 'Space_Marines',
    'Infiltrator Squad': 'Space_Marines', 'Leviathan Dreadnought': 'Space_Marines',
    'Astartes Servitors': 'Space_Marines', 'Captain In Terminator Armour': 'Space_Marines',
    'Brutalis Dreadnought': 'Space_Marines', 'Scout Bike Squad': 'Space_Marines',
    'Roboute Guilliman': 'Space_Marines', 'Thunderhawk Gunship': 'Space_Marines',
    'Vanguard Veteran Squad With Jump Packs': 'Space_Marines',
    'Imperial Space Marine': 'Space_Marines', 'Tycho The Lost': 'Space_Marines',
    'Chief Librarian Mephiston': 'Blood_Angels', 'Astorath': 'Blood_Angels',
    'Sanguinary Priest On Bike': 'Blood_Angels', 'Fortress Of Redemption': 'Space_Marines',
    'Librarian With Jump Pack': 'Space_Marines', 'Lieutenant In Reiver Armour': 'Space_Marines',
    'Lieutenant': 'Space_Marines', 'Land Raider Prometheus': 'Space_Marines',
    'Librarian On Bike': 'Space_Marines', 'Company Veterans On Bikes': 'Space_Marines',
    'Company Champion On Bike': 'Space_Marines', 'Land Speeder Tempest': 'Space_Marines',
    'Land Speeder Tornado': 'Space_Marines', 'Captain In Gravis Armour': 'Space_Marines',
    'Chaplain': 'Space_Marines', 'Ancient On Bike': 'Space_Marines',
    'Terminus Ultra': 'Space_Marines', 'Sicaran Battle Tank': 'Space_Marines',
    'Mastodon': 'Space_Marines', 'Chaplain Venerable Dreadnought': 'Space_Marines',
    'Repressor': 'Space_Marines', 'Valkyrie Sky Talon': 'Space_Marines',
    'Kratos': 'Space_Marines', 'Vindicator Laser Destroyer': 'Space_Marines',
    'Sokar-pattern Stormbird': 'Space_Marines', 'Sicican Omega': 'Space_Marines',
    'Invictor Tactical Warsuit': 'Space_Marines', 'Intercessor Squad': 'Space_Marines',
    'Invader Atv': 'Space_Marines', 'Incursor Squad': 'Space_Marines',
    'Scout Squad': 'Space_Marines', 'Sicican Arcus': 'Space_Marines',
    'Sergeant Telion': 'Space_Marines', 'Land Raider Achilles': 'Space_Marines',
    'Land Raider Proteus': 'Space_Marines', 'Fire Raptor Gunship': 'Space_Marines',
    'Chief Librarian Tigurius': 'Space_Marines', 'Spartan': 'Space_Marines',
    'Deathstorm Drop Pod': 'Space_Marines', 'Deimos Predator': 'Space_Marines',
    'Deredeo Dreadnought': 'Space_Marines', 'Primaris Company Champion': 'Space_Marines',
    'Eliminator Squad': 'Space_Marines', 'Ares Gunship': 'Space_Marines',
    'X-101': 'Space_Marines', 'Terrax-pattern Termite': 'Space_Marines',
    'Impulsor': 'Space_Marines', 'Cultist Firebrand': 'Space_Marines',
    'Death Company Marines': 'Blood_Angels', 'Triumph Of Saint Katherine': 'Adepta_Sororitas',
    'Battle Sanctum': 'Space_Marines', 'Attack Bike Squad': 'Space_Marines',
    'Aleya': 'Space_Marines', 'Inquisitor Draxus': 'Inquisition',
    'The Archivist': 'Space_Marines', 'Aggressor Squad': 'Space_Marines',
    'Adrax Agatone': 'Harlequins', 'Land Speeder Typhoon': 'Space_Marines',
    'Darnath Lysander': 'Space_Marines', 'Tor Garadon': 'Space_Marines',
    'Technoarcheologist': 'Space_Marines', 'Morvenn Vahl': 'Adepta_Sororitas',
    'AESTRED THURGA': 'Adepta_Sororitas', 'AGATHAE DOLAN': 'Adepta_Sororitas',
    'Dogmata': 'Adepta_Sororitas', 'IBRAM GAUNT': 'Astra_Militarum',
    'TANITH GHOST': 'Astra_Militarum',
    
    # SPACE WOLVES
    'Logan Grimnar': 'Space_Wolves', 'Logan Grimnar On Stormrider': 'Space_Wolves',
    'Wolf Lord On Thunderwolf': 'Space_Wolves', 'Krom Dragongaze': 'Space_Wolves',
    'Harald Deathwolf': 'Space_Wolves', 'Ulrik The Slayer': 'Space_Wolves',
    'Wolf Guard Battle Leader In Terminator Armour': 'Space_Wolves',
    'Wolf Guard Battle Leader On Thunderwolf': 'Space_Wolves',
    'Bjorn The Fell-handed': 'Space_Wolves', 'Lukas The Trickster': 'Space_Wolves',
    'Iron Priest On Thunderwolf': 'Space_Wolves', 'Skyclaws': 'Space_Wolves',
    'Long Fangs': 'Space_Wolves', 'Wolf Priest': 'Space_Wolves',
    'Wolf Guard Battle Leader': 'Space_Wolves', 'Wolf Guard Headtakers': 'Space_Wolves',
    'Hunting Wolves': 'Space_Wolves', 'Wulfen Dreadnought': 'Space_Wolves',
    
    # GREY KNIGHTS
    'Kaldor Draigo': 'Grey_Knights', 'Grand Master': 'Grey_Knights',
    'Brother-captain': 'Grey_Knights', 'Brother-captain Stern': 'Grey_Knights',
    'Brotherhood Champion': 'Grey_Knights', 'Brotherhood Terminator Squad': 'Grey_Knights',
    'Paladin Squad': 'Grey_Knights', 'Grey Knights Dreadnought': 'Grey_Knights',
    'Servitors': 'Grey_Knights', 'Brotherhood Techmarine': 'Grey_Knights',
    'Grand Master In Nemesis Dreadknight': 'Grey_Knights',
    'Brotherhood Librarian': 'Grey_Knights', 'Brotherhood Chaplain': 'Grey_Knights',
    'Grey Knights Thunderhawk Gunship': 'Grey_Knights',
    'Grey Knights Relic Razorback': 'Grey_Knights',
    'Grey Knights Terminator Squad': 'Grey_Knights',
    
    # TAU
    'Commander In Coldstar Battlesuit': 'Tau_Empire', "Aun'eshi": 'Tau_Empire',
    "AUN'VA": 'Tau_Empire', 'ETHEREAL GUARD': 'Tau_Empire',
    'Strike Team': 'Tau_Empire', 'Breacher Team': 'Tau_Empire',
    'Krootox Riders': 'Tau_Empire', 'Stealth Battlesuits': 'Tau_Empire',
    'Crisis Battlesuits': 'Tau_Empire', 'Ghostkeel Battlesuit': 'Tau_Empire',
    'Riptide Battlesuit': 'Tau_Empire', 'Piranha': 'Tau_Empire',
    'Devilfish': 'Tau_Empire', 'Razorshark Strike Fighter': 'Tau_Empire',
    'Firesight Team': 'Tau_Empire', 'Longstrike': 'Tau_Empire',
    'Broadside Battlesuits': 'Tau_Empire', 'Stormsurge': 'Tau_Empire',
    'Tidewall Droneport': 'Tau_Empire', 'Tidewall Shieldline': 'Tau_Empire',
    'Tidewall Gunrig': 'Tau_Empire', "Shas'o R'alai": 'Tau_Empire',
    'Xv9 Hazard Battlesuits': 'Tau_Empire', "Y'vahra Battlesuit": 'Tau_Empire',
    "R'varna Battlesuit": 'Tau_Empire', "Ta'unar Supremacy Armour": 'Tau_Empire',
    'Tetras': 'Tau_Empire', 'Tx42 Piranha': 'Tau_Empire',
    'Heavy Gun Drones': 'Tau_Empire', 'Remora Stealth Drones': 'Tau_Empire',
    'Barracuda': 'Tau_Empire', 'Tiger Shark': 'Tau_Empire',
    'Ax-1-0 Tiger Shark': 'Tau_Empire', 'Orca Dropship': 'Tau_Empire',
    'Manta': 'Tau_Empire', 'Remote Sensor Tower': 'Tau_Empire',
    'Drone Sentry Turret': 'Tau_Empire', 'Commander In Crisis Battlesuit': 'Tau_Empire',
    'Commander In Enforcer Battlesuit': 'Tau_Empire',
    'Crisis Sunforge Battlesuits': 'Tau_Empire',
    'Kroot Trail Shaper': 'Tau_Empire', 'Kroot War Shaper': 'Tau_Empire',
    'Kroot Flesh Shaper': 'Tau_Empire', 'Kroot Lone-Spear': 'Tau_Empire',
    'Krootox Rampagers': 'Tau_Empire', 'Ufthak Blackhawk': 'Tau_Empire',
    'GRENADIER': 'Tau_Empire', 'HEAVY WEAPONS TEAM': 'Tau_Empire',
    'QUARTERMASTER REVENANT': 'Tau_Empire', 'MEDICAE SERVITOR': 'Tau_Empire', "AUN’VA": 'Tau_Empire',
    
    # TYRANIDS
    'The Swarmlord': 'Tyranids', 'Genestealers': 'Tyranids',
    'Hormagaunts': 'Tyranids', 'Ripper Swarms': 'Tyranids',
    'Pyrovores': 'Tyranids', 'Raveners': 'Tyranids',
    'Sky-slasher Swarms': 'Tyranids', 'Gargoyles': 'Tyranids',
    'Mucolid Spores': 'Tyranids', 'Spore Mines': 'Tyranids',
    'Carnifexes': 'Tyranids', 'Biovores': 'Tyranids', 'Sporocyst': 'Tyranids',
    'Malanthrope': 'Tyranids', 'Dimachaeron': 'Tyranids',
    'Barbed Hierodule': 'Tyranids', 'Harridan': 'Tyranids',
    'Hierophant': 'Tyranids', 'Scythed Hierodule': 'Tyranids',
    'Winged Hive Tyrant': 'Tyranids', 'Parasite Of Mortrex': 'Tyranids',
    'Tyranid Warriors With Melee Bio-weapons': 'Tyranids',
    'Tyranid Warriors With Ranged Bio-weapons': 'Tyranids',
    "Von Ryan's Leapers": 'Tyranids', 'Winged Tyranid Prime': 'Tyranids',
    'Norn Emissary': 'Tyranids', 'Norn Assimilator': 'Tyranids',
    'Lhykhis': 'Tyranids', 'Ravener Prime': 'Tyranids',
    'Servitor Underseer': 'Tyranids', 'Combat Servitors and Gun Servitors': 'Tyranids',
    
    # NECRONS
    'Patriarch': 'Genestealer_Cult', 'Primus': 'Genestealer_Cult',
    'Purestrain Genestealers': 'Genestealer_Cult',
    'Imotekh The Stormlord': 'Necrons', 'Lord': 'Necrons',
    'Orikan The Diviner': 'Necrons', "C'tan Shard of the Nightbringer": 'Necrons',
    "C'tan Shard of the Deceiver": 'Necrons', 'Canoptek Spyders': 'Necrons',
    "Transcendent C'tan": 'Necrons', 'Canoptek Tomb Stalker': 'Necrons',
    'Canoptek Acanthrites': 'Necrons', 'Canoptek Tomb Sentinel': 'Necrons',
    'Night Shroud': 'Necrons', 'Sentry Pylon': 'Necrons',
    'Tesseract Ark': 'Necrons', 'Gauss Pylon': 'Necrons',
    'SZAREKH': 'Necrons', 'TRIARCHAL MENHIR': 'Necrons',
    'Convergence Of Dominion': 'Necrons', 'Tomb Citadel Walls': 'Necrons',
    'Guardian Drone': 'Necrons', 'Skorpekh Lord': 'Necrons',
    'Skorpekh Destroyers': 'Necrons', 'Lokhust Lord': 'Necrons',
    'Technomancer': 'Necrons', 'Hexmark Destroyer': 'Necrons',
    "C'tan Shard of the Void Dragon": 'Necrons', 'Lokhust Destroyers': 'Necrons',
    'Canoptek Macrocytes': 'Necrons', 'Canoptek Tomb Crawlers': 'Necrons',
    'Geomancer': 'Necrons', 'Nekrosor Ammentar': 'Necrons',
    
    # CRAFTWORLDS / AELDARI
    'Warlock Conclave': 'Craftworlds', 'Warlock': 'Craftworlds',
    'GUARDIAN DEFENDER': 'Craftworlds', 'HEAVY WEAPON PLATFORM': 'Craftworlds',
    'STORM GUARDIAN': 'Craftworlds', "SERPENT'S SCALE PLATFORM": 'Craftworlds',
    'DIRE AVENGER': 'Craftworlds', 'DIRE AVENGER EXARCH': 'Craftworlds',
    'HOWLING BANSHEE': 'Craftworlds', 'HOWLING BANSHEE EXARCH': 'Craftworlds',
    'STRIKING SCORPION': 'Craftworlds', 'STRIKING SCORPION EXARCH': 'Craftworlds',
    'FIRE DRAGON': 'Craftworlds', 'FIRE DRAGON EXARCH': 'Craftworlds',
    'SWOOPING HAWK': 'Craftworlds', 'SWOOPING HAWK EXARCH': 'Craftworlds',
    'WARP SPIDER': 'Craftworlds', 'WARP SPIDER EXARCH': 'Craftworlds',
    'SHINING SPEAR': 'Craftworlds', 'SHINING SPEAR EXARCH': 'Craftworlds',
    'Vypers': 'Craftworlds', 'DARK REAPER': 'Craftworlds',
    'DARK REAPER EXARCH': 'Craftworlds', 'War Walkers': 'Craftworlds',
    'SHADOW SPECTRE': 'Craftworlds', 'SHADOW SPECTRE EXARCH': 'Craftworlds',
    'Wasp Assault Walker': 'Craftworlds', 'Wraithseer': 'Craftworlds',
    'Hornet': 'Craftworlds', 'Warp Hunter': 'Craftworlds', 'Lynx': 'Craftworlds',
    'Scorpion': 'Craftworlds', 'Cobra': 'Craftworlds', 'Nightwing': 'Craftworlds',
    'Phoenix': 'Craftworlds', 'Vampire Raider': 'Craftworlds',
    'Vampire Hunter': 'Craftworlds', 'Skathach Wraithknight': 'Craftworlds',
    'Revenant Titan': 'Craftworlds', 'Irillyth': 'Craftworlds',
    'Phantom Titan': 'Craftworlds', 'Autarch Wayleaper': 'Craftworlds',
    'Bonesinger': 'Craftworlds', 'D-cannon Platform': 'Craftworlds',
    'Shadow Weaver Platform': 'Craftworlds', 'Vibro Cannon Platform': 'Craftworlds',
    
    # DRUKHARI
    'INCUBI': 'Drukhari', 'KLAIVEX': 'Drukhari', 'Grotesques': 'Drukhari',
    'BEASTMASTER': 'Drukhari', 'CLAWED FIEND': 'Drukhari', 'KHYMERAE': 'Drukhari',
    'RAZORWING FLOCK': 'Drukhari', 'Reaper': 'Drukhari', 'Tantalus': 'Drukhari',
    'Court Of The Archon': 'Drukhari', 'Lady Malys': 'Drukhari',
    'Hand of the Archon': 'Drukhari', 'Shalaxi Helbane': 'Drukhari',
    
    # HARLEQUINS
    'Knarloc Riders': 'Harlequins', 'Great Knarloc': 'Harlequins',
    'Firestorm': 'Harlequins', 'MINKA LESK': 'Harlequins',
    "LESK'S VETERANS": 'Harlequins',
    
    # CORSAIRS / YNNARI
    'Corsair Cloud Dancer Band': 'Craftworlds', 'Corsair Reaver Band': 'Craftworlds',
    'Corsair Skyreaver Band': 'Craftworlds', 'Corsair Voidreavers': 'Craftworlds',
    'Corsair Voidscarred': 'Craftworlds', 'Ynnari Archon': 'Ynnari',
    'Ynnari Succubus': 'Ynnari', 'Ynnari Kabalite Warriors': 'Ynnari',
    'Ynnari Wyches': 'Ynnari', 'Ynnari Reavers': 'Ynnari',
    'Ynnari Raider': 'Ynnari', 'Corsair Skyreavers': 'Craftworlds',
    'Starfangs': 'Craftworlds',
    
    # ASTRA MILITARUM
    'Leman Russ Commander': 'Astra_Militarum', 'Heavy Weapons Squad': 'Astra_Militarum',
    'Scout Sentinels': 'Astra_Militarum', 'Armoured Sentinels': 'Astra_Militarum',
    'Leman Russ Battle Tank': 'Astra_Militarum', 'Iron Hand Straken': 'Astra_Militarum',
    'Commissar': 'Astra_Militarum', 'TEMPESTUS SCION': 'Astra_Militarum',
    'TEMPESTOR PRIME': 'Astra_Militarum', 'Taurox Prime': 'Astra_Militarum',
    'Ogryn Squad': 'Astra_Militarum', 'Bullgryn Squad': 'Astra_Militarum',
    'Atlas Recovery Vehicle': 'Astra_Militarum',
    'Salamander Command Vehicle': 'Astra_Militarum',
    'Hades Breaching Drill': 'Astra_Militarum', 'Centaur Light Carrier': 'Astra_Militarum',
    'Trojan Support Vehicle': 'Astra_Militarum', 'Salamander Scout Vehicle': 'Astra_Militarum',
    'Tauros Assault Vehicle': 'Astra_Militarum', 'Tauros Venator': 'Astra_Militarum',
    'Armageddon-pattern Medusa': 'Astra_Militarum', 'Colossus': 'Astra_Militarum',
    'Cyclops Demolition Vehicle': 'Astra_Militarum', 'Earthshaker Carriage Battery': 'Astra_Militarum',
    'Griffon Mortar Carrier': 'Astra_Militarum', 'Heavy Mortar Team': 'Astra_Militarum',
    'Heavy Quad Launcher Team': 'Astra_Militarum', 'Hydra Platform': 'Astra_Militarum',
    'Malcador Annihilator': 'Astra_Militarum', 'Malcador Defender': 'Astra_Militarum',
    'Malcador': 'Astra_Militarum', 'Malcador Infernus': 'Astra_Militarum',
    'Manticore Platform': 'Astra_Militarum', 'Medusa Carriage Battery': 'Astra_Militarum',
    'Rapier Laser Destroyer Battery': 'Astra_Militarum', 'Sabre Weapons Battery': 'Astra_Militarum',
    'Sentinel Powerlifter': 'Astra_Militarum', 'Stygies Destroyer Tank Hunter': 'Astra_Militarum',
    'Tarantula Battery': 'Astra_Militarum', 'Stormblade': 'Astra_Militarum',
    'Arkurian Stormhammer': 'Astra_Militarum', 'Crassus': 'Astra_Militarum',
    'Dominus Armoured Siege Bombard': 'Astra_Militarum',
    'Gorgon Heavy Transport': 'Astra_Militarum', 'Macharius': 'Astra_Militarum',
    'Macharius Omega': 'Astra_Militarum', 'Macharius Vanquisher': 'Astra_Militarum',
    'Macharius Vulcan': 'Astra_Militarum', 'Marauder Bomber': 'Astra_Militarum',
    'Marauder Destroyer': 'Astra_Militarum', 'Minotaur': 'Astra_Militarum',
    'Praetor': 'Astra_Militarum', 'Valdor': 'Astra_Militarum',
    'Aquila Lander': 'Astra_Militarum', 'Arvus Lighter': 'Astra_Militarum',
    'Avenger Strike Fighter': 'Astra_Militarum', 'Voss-pattern Lightning': 'Astra_Militarum',
    'Vendetta Gunship': 'Astra_Militarum', 'Vulture Gunship': 'Astra_Militarum',
    'Imperial Fortress Walls': 'Astra_Militarum', 'Primaris Redoubt': 'Astra_Militarum',
    'Ministorum Priest': 'Astra_Militarum', 'Munitorum Servitors': 'Astra_Militarum',
    'Ogryn Bodyguard': 'Astra_Militarum', 'Elysian Sniper Squad': 'Astra_Militarum',
    'Death Rider Commissar': 'Astra_Militarum', 'Storm Chimera': 'Astra_Militarum',
    'Voidsmen-at-arms': 'Astra_Militarum', 'Valerian': 'Astra_Militarum',
    'Knight-centura': 'Astra_Militarum', 'Prosecutors': 'Astra_Militarum',
    'Vigilators': 'Astra_Militarum', 'Witchseekers': 'Astra_Militarum',
    'Anathema Psykana Rhino': 'Astra_Militarum', 'CADIAN COMMANDER': 'Astra_Militarum',
    'CADIAN VETERAN GUARDSMAN': 'Astra_Militarum', 'Regimental Attachés': 'Astra_Militarum',
    'Death Korps Of Krieg': 'Astra_Militarum', 'Kasrkin': 'Astra_Militarum',
    'Attilan Rough Riders': 'Astra_Militarum', 'Aegis Defence Line': 'Astra_Militarum',
    'Earthshaker Platform': 'Astra_Militarum', 'Death Korps Of Krieg': 'Astra_Militarum',
    'Krieg Combat Engineers': 'Astra_Militarum', 'Catachan Heavy Weapons Squad': 'Astra_Militarum',
    'HEAVY WEAPONS GUNNER': 'Astra_Militarum', 'FIRE COORDINATOR': 'Astra_Militarum',
    'Artillery Team': 'Astra_Militarum', 'Death Riders': 'Astra_Militarum',
    'Leman Russ Eradicator': 'Astra_Militarum', 'Leman Russ Executioner': 'Astra_Militarum',
    'Leman Russ Exterminator': 'Astra_Militarum', 'Leman Russ Punisher': 'Astra_Militarum',
    'Leman Russ Vanquisher': 'Astra_Militarum', 'CATACHAN COMMANDER': 'Astra_Militarum',
    'VETERAN GUARDSMAN': 'Astra_Militarum', 'Lord Marshal Dreir': 'Astra_Militarum',
    'Rogal Dorn Tank Commander': 'Astra_Militarum', 'Elysian Drop Sentinel': 'Astra_Militarum',
    'Imperial Navy Breachers': 'Astra_Militarum',
    'FARSTALKERS & KILL-BROKER': 'Astra_Militarum', 'KROOT HOUNDS': 'Astra_Militarum',
    'LORD COMMISSAR': 'Astra_Militarum', 'Benefictus': 'Astra_Militarum',
    'Celestian Sacresant Aveline': 'Adepta_Sororitas',
    'Acolyte Hybrids With Hand Flamers': 'Genestealer_Cult',
    
    # ADEPTUS MECHANICUS
    'Tech-priest Dominus': 'Adeptus_Mechanicus', 'Tech-priest Enginseer': 'Adeptus_Mechanicus',
    'Kataphron Breachers': 'Adeptus_Mechanicus', 'Fulgurite Electro-priests': 'Adeptus_Mechanicus',
    'Corpuscarii Electro-priests': 'Adeptus_Mechanicus', 'Kastelan Robots': 'Adeptus_Mechanicus',
    'Cybernetica Datasmith': 'Adeptus_Mechanicus', 'Ironstrider Ballistarii': 'Adeptus_Mechanicus',
    'Sydonian Dragoons': 'Adeptus_Mechanicus', 'Acastus Knight Porphyrion': 'Adeptus_Mechanicus',
    'Cerastus Knight Acheron': 'Adeptus_Mechanicus', 'Cerastus Knight Atrapos': 'Adeptus_Mechanicus',
    'Cerastus Knight Castigator': 'Adeptus_Mechanicus', 'Cerastus Knight Lancer': 'Adeptus_Mechanicus',
    'Questoris Knight Magaera': 'Adeptus_Mechanicus', 'Questoris Knight Styrix': 'Adeptus_Mechanicus',
    'Warhound Titan': 'Adeptus_Mechanicus', 'Reaver Titan': 'Adeptus_Mechanicus',
    'Warlord Titan': 'Adeptus_Mechanicus', 'Telemon Heavy Dreadnought': 'Adeptus_Mechanicus',
    'Armiger Warglaive': 'Adeptus_Mechanicus', 'Scout Sniper Squad': 'Adeptus_Mechanicus',
    'Sergeant Chronus': 'Adeptus_Mechanicus', 'Secutarii Hoplites': 'Adeptus_Mechanicus',
    'Secutarii Peltasts': 'Adeptus_Mechanicus', 'Techmarine On Bike': 'Adeptus_Mechanicus',
    'Tectonic Fragdrill': 'Adeptus_Mechanicus', 'Seraptek Heavy Construct': 'Adeptus_Mechanicus',
    'Inquisitor Eisenhorn': 'Inquisition', 'Tech-priest Manipulus': 'Adeptus_Mechanicus',
    'Sydonian Skatros': 'Adeptus_Mechanicus', 'Thulia Ghuld': 'Adeptus_Mechanicus',
    'Hastarii Exterminators': 'Adeptus_Mechanicus', 'Hastarii Fusiliers': 'Adeptus_Mechanicus',
    
    # IMPERIAL KNIGHTS
    'Knight Despoiler': 'Chaos_Knights', 'Knight Tyrant': 'Chaos_Knights',
    'Chaos Acastus Knight Asterius': 'Chaos_Knights', 'War Dog Moirax': 'Chaos_Knights',
    'Acastus Knight Asterius': 'Imperial_Knights', 'Falchion': 'Imperial_Knights',
    'War Dog Executioner': 'Imperial_Knights', 'War Dog Stalker': 'Imperial_Knights',
    'War Dog Karnivore': 'Imperial_Knights', 'War Dog Brigand': 'Imperial_Knights',
    'War Dog Huntsman': 'Imperial_Knights', 'Knight Abominant': 'Chaos_Knights',
    'Knight Ruinator': 'Chaos_Knights', 'Execrator': 'Chaos_Knights',
    'Knight Defender': 'Imperial_Knights', 'Armiger Moirax': 'Imperial_Knights',
    'Sir Hekhtur': 'Imperial_Knights', 'Crusade Ancient': 'Imperial_Knights',
    'Buri Aegnirssen': 'Imperial_Knights', 'Memnyr Strategist': 'Imperial_Knights',
    'Arkanyst Evaluator': 'Imperial_Knights', 'Knight Destrier': 'Imperial_Knights',
    'Greater Daemon': 'Chaos_Daemons', 'Lesser Daemons': 'Chaos_Daemons',
    'Abomination': 'Chaos_Daemons', 'Daemonic Beasts': 'Chaos_Daemons',
    
    # INQUISITION / ASSASSINS
    'Inquisitor': 'Inquisition', 'Daemonhost': 'Inquisition',
    'Imperial Rhino': 'Inquisition', 'Tempestus Aquilons': 'Astra_Militarum',
    'INQUISITOR OSTROMANDEUS': 'Inquisition', 'STENTOR-I-52': 'Inquisition',
    'Rein And Raus': 'Imperial_Assassins', 'Amallyn Shadowguide': 'Imperial_Assassins',
    'Rogue Psyker': 'Inquisition', 'Negavolt Cultists': 'Inquisition',
    'Spindle Drones': 'Adeptus_Mechanicus',
    
    # ADEPTA SORORITAS
    'CELESTINE': 'Adepta_Sororitas', 'GEMINAE SUPERIA': 'Adepta_Sororitas',
    'Seraphim Squad': 'Adepta_Sororitas', 'REPENTIA SUPERIOR': 'Adepta_Sororitas',
    'SISTERS REPENTIA': 'Adepta_Sororitas', 'Dominion Squad': 'Adepta_Sororitas',
    'Sororitas Rhino': 'Adepta_Sororitas', 'Bastion': 'Adepta_Sororitas',
    'Vengeance Weapon Battery': 'Adepta_Sororitas', 'Firestorm Redoubt': 'Adepta_Sororitas',
    'Plasma Obliterator': 'Adepta_Sororitas', 'Macro-cannon Aquila Strongpoint': 'Adepta_Sororitas',
    'Vortex Missile Strongpoint': 'Adepta_Sororitas', 'Void Shield Generator': 'Adepta_Sororitas',
    'Skyshield Landing Pad': 'Adepta_Sororitas', 'NOVITIATE SUPERIOR': 'Adepta_Sororitas',
    'SISTER NOVITIATE': 'Adepta_Sororitas', 'EPHRAEL STERN': 'Adepta_Sororitas',
    'KYGANIL OF THE BLOODY TEARS': 'Adepta_Sororitas', 'Apothecary': 'Adepta_Sororitas',
    'Ancient': 'Adepta_Sororitas', 'GRIMALDUS': 'Space_Marines',
    'CENOBYTE SERVITOR': 'Space_Marines', 'Castellan': 'Space_Marines',
    "Emperor's Champion": 'Space_Marines', 'Marshal': 'Space_Marines',
    'Sword Brethren Squad': 'Space_Marines',
    
    # CHAOS SPACE MARINES
    'FABIUS BILE': 'Chaos_Space_Marines', 'SURGEON ACOLYTE': 'Chaos_Space_Marines',
    'Chaos Lord In Terminator Armour': 'Chaos_Space_Marines',
    'Chaos Lord On Bike': 'Chaos_Space_Marines',
    'Chaos Lord On Juggernaut': 'Chaos_Space_Marines',
    'Chaos Lord On Disc Of Tzeentch': 'Chaos_Space_Marines',
    'Chaos Lord On Palanquin Of Nurgle': 'Chaos_Space_Marines',
    'Chaos Lord On Steed Of Slaanesh': 'Chaos_Space_Marines',
    'DARK APOSTLE': 'Chaos_Space_Marines', 'DARK DISCIPLE': 'Chaos_Space_Marines',
    'Heretic Astartes Daemon Prince': 'Chaos_Space_Marines', 'Sorcerer': 'Chaos_Space_Marines',
    'Sorcerer In Terminator Armour': 'Chaos_Space_Marines', 'Sorcerer On Bike': 'Chaos_Space_Marines',
    'Sorcerer On Disc Of Tzeentch': 'Chaos_Space_Marines',
    'Sorcerer On Palanquin Of Nurgle': 'Chaos_Space_Marines',
    'Sorcerer On Steed Of Slaanesh': 'Chaos_Space_Marines', 'Cultist Mob': 'Chaos_Space_Marines',
    'Chaos Terminator Squad': 'Chaos_Space_Marines',
    'Chaos Predator Destructor': 'Chaos_Space_Marines', 'Chaos Vindicator': 'Chaos_Space_Marines',
    'Obliterators': 'Chaos_Space_Marines', 'Havocs': 'Chaos_Space_Marines',
    'Khorne Lord Of Skulls': 'Chaos_Space_Marines',
    'Rubric Marine': 'Thousand_Sons', 'Aspiring Sorcerer': 'Thousand_Sons',
    'Magnus The Red': 'Thousand_Sons', 'Exalted Sorcerer': 'Thousand_Sons',
    'Scarab Occult Terminator': 'Thousand_Sons', 'Scarab Occult Sorcerer': 'Thousand_Sons',
    'Daemon Prince of Tzeentch': 'Thousand_Sons',
    'Death Guard Chaos Lord': 'Death_Guard',
    'Death Guard Chaos Lord In Terminator Armour': 'Death_Guard',
    'Death Guard Sorcerer In Terminator Armour': 'Death_Guard',
    'Death Guard Cultists': 'Death_Guard', 'Death Guard Possessed': 'Death_Guard',
    'Chaos Predator Annihilator': 'Death_Guard', 'Typhus': 'Death_Guard',
    'Foetid Bloat-drone': 'Death_Guard',
    'Chaos Cerastus Knight Acheron': 'Chaos_Knights',
    'Chaos Cerastus Knight Lancer': 'Chaos_Knights',
    'Chaos Cerastus Knight Castigator': 'Chaos_Knights',
    'Chaos Cerastus Knight Atrapos': 'Chaos_Knights',
    'Chaos Questoris Knight Magaera': 'Chaos_Knights',
    'Chaos Acastus Knight Porphyrion': 'Chaos_Knights',
    'Chaos Questoris Knight Styrix': 'Chaos_Knights',
    'Karanak': 'Chaos_Daemons', 'Rendmaster On Blood Throne': 'Chaos_Daemons',
    'Bloodletters': 'Chaos_Daemons', 'Bloodcrushers': 'Chaos_Daemons',
    'Skull Cannon': 'Chaos_Daemons', 'Flamers': 'Chaos_Daemons',
    'Exalted Flamer': 'Chaos_Daemons', 'Screamers': 'Chaos_Daemons',
    'Burning Chariot': 'Chaos_Daemons', 'Plaguebearers': 'Chaos_Daemons',
    'Nurglings': 'Chaos_Daemons', 'Beasts Of Nurgle': 'Chaos_Daemons',
    'Plague Drones': 'Chaos_Daemons', 'The Masque Of Slaanesh': 'Chaos_Daemons',
    'Keeper Of Secrets': 'Chaos_Daemons', 'Tranceweaver': 'Chaos_Daemons',
    'Herald Of Slaanesh On Steed Of Slaanesh': 'Chaos_Daemons',
    'Tormentbringer On Exalted Seeker Chariot': 'Chaos_Daemons',
    'Daemonettes': 'Chaos_Daemons', 'Hellflayer': 'Chaos_Daemons',
    'Seeker Chariot': 'Chaos_Daemons', 'Exalted Seeker Chariot': 'Chaos_Daemons',
    "Be'lakor": 'Chaos_Daemons', 'Daemon Prince of Chaos': 'Chaos_Daemons',
    'Furies': 'Chaos_Daemons', 'Cerberus': 'Chaos_Daemons',
    'Exalted Champion': 'Chaos_Space_Marines', 'Plague Surgeon': 'Death_Guard',
    'Tallyman': 'Death_Guard', 'Deathshroud Terminators': 'Death_Guard',
    'Myphitic Blight-haulers': 'Death_Guard', 'Mukaali Riders': 'Death_Guard',
    'Grot Bomm Launcha': 'Orks', 'Attack Fighta': 'Orks',
    'Fighta-bommer': 'Orks', 'Deff Rolla Battle Fortress': 'Orks',
    'Kill Krusha': 'Orks', 'Raven Strike Fighter': 'Orks',
    'Blood Slaughterer': 'Chaos_Space_Marines', 'Greater Blight Drone': 'Death_Guard',
    'Decimator': 'Chaos_Knights', 'Kytan Ravager': 'Chaos_Knights',
    'Greater Brass Scorpion': 'Chaos_Knights', 'Chaos Deimos Predator': 'Chaos_Space_Marines',
    'Dreadclaw Drop Pod': 'Chaos_Space_Marines', 'Kharybdis Assault Claw': 'Chaos_Space_Marines',
    'Hell Blade': 'Chaos_Space_Marines', 'Hell Talon': 'Chaos_Space_Marines',
    'Chaos Thunderhawk': 'Chaos_Space_Marines', "An'ggrath The Unbound": 'Chaos_Daemons',
    'Zarakynel': 'Chaos_Daemons', "Aetaos'rau'keres": 'Chaos_Daemons',
    'Plague Toads': 'Chaos_Daemons', 'Pox Riders': 'Chaos_Daemons',
    'Spined Chaos Beast': 'Chaos_Daemons', 'Giant Chaos Spawn': 'Chaos_Daemons',
    'Scabeiathrax The Bloated': 'Death_Guard', 'Changecaster': 'Thousand_Sons',
    'Fateskimmer': 'Thousand_Sons', 'Fluxmaster': 'Thousand_Sons',
    'Feculent Gnarlmaw': 'Death_Guard', 'Master Of Possession': 'Chaos_Space_Marines',
    'Lord Discordant On Helstalker': 'Chaos_Space_Marines',
    'Master Of Executions': 'Chaos_Space_Marines', 'Venomcrawler': 'Chaos_Space_Marines',
    'Noctilith Crown': 'Necrons', 'Skull Altar': 'Chaos_Daemons',
    'Fellgor Beastmen': 'Chaos_Space_Marines', 'Traitor Guardsmen Squad': 'Chaos_Space_Marines',
    'MUTANT': 'Chaos_Space_Marines', 'TORMENT': 'Chaos_Space_Marines',
    'NIGHTMARE HULK': 'Chaos_Space_Marines', 'GELLERPOX MUTANTS': 'Death_Guard',
    'Mutoid Vermin': 'Chaos_Space_Marines', 'TRAITOR ENFORCER': 'Chaos_Space_Marines',
    'TRAITOR OGRYN': 'Chaos_Space_Marines', 'Traitor Guardsmen Squad': 'Chaos_Space_Marines',
    'OGRYN PACK MASTER': 'Chaos_Space_Marines', 'CHAOS MAULER HOUND': 'Chaos_Space_Marines',
    'Renegade Ogryn Brutes': 'Chaos_Space_Marines', 'Renegade Plague Ogryns': 'Death_Guard',
    'Renegade Enforcer': 'Chaos_Space_Marines', 'Renegade Heavy Weapons Squad': 'Chaos_Space_Marines',
    'Legionaries': 'Chaos_Space_Marines', 'Blue Horrors': 'Chaos_Daemons',
    'PINK HORROR': 'Chaos_Daemons', 'BLUE HORROR/BRIMSTONE HORROR': 'Chaos_Daemons',
    'Fulgrim': 'Chaos_Space_Marines', 'Lord Exultant': 'Chaos_Space_Marines',
    'Tormentors': 'Chaos_Space_Marines', 'Infractors': 'Chaos_Space_Marines',
    'Lucius the Eternal': 'Chaos_Space_Marines', 'Lord Kakophonist': 'Chaos_Space_Marines',
    'Daemon Prince of Slaanesh': 'Chaos_Daemons',
    'Daemon Prince of Slaanesh with Wings': 'Chaos_Daemons',
    'Noise Marines': 'Chaos_Space_Marines', 'Flawless Blades': 'Chaos_Space_Marines',
    'Tormentbringer': 'Chaos_Daemons', 'Lord of Poxes': 'Death_Guard',
    'Plague Drones': 'Death_Guard', 'Beasts of Nurgle': 'Chaos_Daemons',
    'Daemon Prince of Tzeentch with Wings': 'Thousand_Sons',
    'Sekhetar Robots': 'Thousand_Sons',
    'Tzaangor Enlightened with Fatecaster Greatbows': 'Thousand_Sons',
    "Syll'esske": 'Chaos_Daemons', "An'ggrath The Unbound": 'Chaos_Daemons',
    'Garlon Souleater': 'Chaos_Space_Marines', 'Garreon The Corpsemaster': 'Chaos_Space_Marines',
    'Katar Garrix': 'Chaos_Space_Marines', 'Captain Sargotta': 'Chaos_Space_Marines',
    'The Enforcer': 'Chaos_Space_Marines', 'KÔhl': 'Chaos_Space_Marines',
    'Vashtorr The Arkifane': 'Chaos_Daemons', 'Angron': 'Chaos_Space_Marines',
    'KhÔrn The Betrayer': 'Chaos_Daemons', 'Lord Invocatus': 'Chaos_Space_Marines',
    'Daemon Prince of Khorne': 'Chaos_Daemons', 'Lord On Juggernaut': 'Chaos_Daemons',
    'Jakhals': 'Chaos_Space_Marines', 'Eightbound': 'Chaos_Space_Marines',
    'Exalted Eightbound': 'Chaos_Space_Marines', 'Ancient In Terminator Armour': 'Chaos_Space_Marines',
    'Chaplain Cassius': 'Space_Marines', "Lion El'jonson": 'Dark_Angels',
    'Vigilant Squad': 'Dark_Angels', 'Subductor Squad': 'Dark_Angels',
    'Exaction Squad': 'Dark_Angels', 'Screamer-killer': 'Dark_Angels',
    'Damned Legionnaires': 'Dark_Angels', 'Janus Draik': 'Dark_Angels',
    'Neyam Shai Murad': 'Dark_Angels', 'Navigator': 'Inquisition',
    'Ur-025': 'Inquisition', 'Inquisitorial Agents': 'Inquisition',
    
    # DEATHWATCH
    'Deathwatch Veterans': 'Deathwatch', 'KILL TEAM VETERAN': 'Deathwatch',
    'KILL TEAM BIKER': 'Deathwatch', 'KILL TEAM TERMINATOR': 'Deathwatch',
    'Deathwatch Terminator Squad': 'Deathwatch', 'Veteran Bike Squad': 'Deathwatch',
    'KILL TEAM VETERANS': 'Deathwatch', 'KILL TEAM INTERCESSOR': 'Deathwatch',
    'KILL TEAM OUTRIDER': 'Deathwatch', 'Indomitor Kill Team': 'Deathwatch',
    'Spectrus Kill Team': 'Deathwatch', 'Fortis Kill Team': 'Deathwatch',
    'KILL TEAM SERGEANT WITH JUMP PACK AND KILL TEAM INTERCESSORS WITH JUMP PACKS': 'Deathwatch',
    'KILL TEAM HEAVY INTERCESSORS WITH JUMP PACKS': 'Deathwatch',
    'KILL TEAM SERGEANT, DEATHWATCH VETERAN': 'Deathwatch',
    'GRAVIS VETERAN': 'Deathwatch', 'KILL TEAM SERGEANT, DEATHWATCH VETERAN': 'Deathwatch',
    'GRAVIS VETERAN': 'Deathwatch', 'KILL TEAM SERGEANT WITH JUMP PACK AND KILL TEAM INTERCESSORS WITH JUMP PACKS': 'Deathwatch',
    
    # ADEPTUS CUSTODES
    'Trajann Valoris': 'Adeptus_Custodes', 'Shield-captain': 'Adeptus_Custodes',
    'Shield-captain In Allarus Terminator Armour': 'Adeptus_Custodes',
    'Shield-captain On Dawneagle Jetbike': 'Adeptus_Custodes',
    'Bloodmaster': 'Adeptus_Custodes', 'Skullmaster': 'Adeptus_Custodes',
    'Contemptor-achillus Dreadnought': 'Adeptus_Custodes',
    'Caladius Grav-tank': 'Adeptus_Custodes', 'Coronus Grav-carrier': 'Adeptus_Custodes',
    'Aquilon Custodians': 'Adeptus_Custodes',
    'Custodian Guard With Adrasite And Pyrithite Spears': 'Adeptus_Custodes',
    'Agamatus Custodians': 'Adeptus_Custodes', 'Venatari Custodians': 'Adeptus_Custodes',
    'Pallas Grav-attack': 'Adeptus_Custodes', 'Sagittarum Custodians': 'Adeptus_Custodes',
    'Orion Assault Dropship': 'Adeptus_Custodes', 'Achilles Ridgerunners': 'Adeptus_Custodes',
    'ATALAN JACKAL': 'Adeptus_Custodes', 'ATALAN WOLFQUAD': 'Adeptus_Custodes',
    'Contemptor-galatus Dreadnought': 'Adeptus_Custodes',
    'Captain In Phobos Armour': 'Adeptus_Custodes', 'Captain On Bike': 'Adeptus_Custodes',
    'Kayvaan Shrike': 'Adeptus_Custodes', "Kor'sarro Khan": 'Craftworlds',
    'Land Raider Helios': 'Adeptus_Custodes', 'OUTRIDER': 'Adeptus_Custodes',
    'Pedro Kantor': 'Space_Marines', 'Predator Annihilator': 'Space_Marines',
    'Predator Destructor': 'Space_Marines', 'Reiver Squad': 'Space_Marines',
    'Relic Contemptor Dreadnought': 'Space_Marines', 'Relic Razorback': 'Space_Marines',
    'Repulsor Executioner': 'Space_Marines', 'Repulsor': 'Space_Marines',
    'Thunderhawk Transporter': 'Space_Marines', 'Tyrannic War Veterans': 'Space_Marines',
    "Vulkan He'stan": 'Salamanders', 'Whirlwind Scorpius': 'Space_Marines',
    'Xiphon Interceptor': 'Space_Marines',
    'Heretic Astartes Daemon Prince With Wings': 'Chaos_Space_Marines',
    'Daemon Prince of Nurgle with Wings': 'Chaos_Daemons',
    'Exalted Sorcerer on Disc of Tzeentch': 'Thousand_Sons',
    'Daemon Prince of Khorne with Wings': 'Chaos_Daemons',
    'Daemon Prince Of Chaos With Wings': 'Chaos_Daemons',
    'Inquisitor In Terminator Armour': 'Inquisition', 'ROGUE TRADER': 'Inquisition',
    'OTHER MODELS': 'Space_Marines', 'Death Company Captain': 'Blood_Angels',
    'Death Company Dreadnought with Magna-grapple': 'Blood_Angels',
    'Death Company Marines with Boltguns': 'Blood_Angels',
    'Death Company Marines with Boltguns and Jump Packs': 'Blood_Angels',
    'Death Company Marines With Jump Packs': 'Blood_Angels',
    'Sanguinary Priest With Jump Packs': 'Blood_Angels',
    'Assault Intercessors With Jump Packs': 'Space_Marines',
    'Deathwatch Veterans': 'Deathwatch',
    
    # LEAGUES OF VOTANN
    'Einhyr Champion': 'Leagues_of_Votann', 'GRIMNYR': 'Leagues_of_Votann',
    'CORV': 'Leagues_of_Votann', 'BRÈKHYR IRON-MASTER': 'Leagues_of_Votann',
    'IRONKIN ASSISTANT': 'Leagues_of_Votann', 'E-COG': 'Leagues_of_Votann',
    'Hearthkyn Warriors': 'Leagues_of_Votann', 'Einhyr Hearthguard': 'Leagues_of_Votann',
    'Cthonian Beserks': 'Leagues_of_Votann', 'Hernkyn Pioneers': 'Leagues_of_Votann',
    'Sagitaur': 'Leagues_of_Votann', 'Br¶khyr Thunderkyn': 'Leagues_of_Votann',
    'Hekaton Land Fortress': 'Leagues_of_Votann',
    'Ursula Creed': 'Leagues_of_Votann',
    
    # WARHAMMER WORLD / GENERIC
    'NEOPHYTES': 'Genestealer_Cult', 'Warbringer Nemesis Titan': 'Imperial_Knights',
    'Astraeus': 'Craftworlds', 'Fellblade': 'Space_Marines',
    
    # DUPLICATES / GENERIC ENTRIES - assign based on context
    'MARNEUS CALGAR': 'Space_Marines', 'Cato Sicarius': 'Space_Marines',
    'Victrix Honour Guard': 'Supplement_Marines', 'Captain Titus': 'Space_Marines',
    'Ancient Gadriel, Veteran Sergeant Metaurus': 'Space_Marines',
    'Gaius Silva, Aemelia Minervas, Dainal Kornelius, Lucia Vestha': 'Space_Marines',
    'Celestian Insidiants': 'Adepta_Sororitas',
    'GARLON SOULEATER, GARREON THE CORPSEMASTER, KATAR GARRIX': 'Chaos_Space_Marines',
    'CAPTAIN SARGOTTA, THE ENFORCER': 'Chaos_Space_Marines',
    'Ferren Areios': 'Supplement_Marines',
    
    # Additional entries
    'Chaplain On Bike': 'Space_Marines', 'Sicican Venator': 'Space_Marines',
    'Typhon': 'Space_Marines', 'War Dog Moirax': 'Imperial_Knights',
    'Iron Priest On Thunderwolf': 'Space_Wolves',
    "Ta'unar Supremacy Armour" : 'Tau_Empire',
    "SERPENT'S SCALE PLATFORM": 'Craftworlds',
    'CLAWED FIEND': 'Drukhari',
    "Be'lakor": 'Chaos_Daemons',
    'Sicaran Omega': 'Space_Marines',
    'Ambull': 'Space_Marines',
    "Big'ed Bossbunka": 'Orks',
    'Ûthar The Destined': 'Leagues_of_Votann',
    'Kâhl': 'Leagues_of_Votann',
    'Brôkhyr Thunderkyn': 'Leagues_of_Votann',
    "Lion El'jonson": 'Dark_Angels',
    "Von Ryan's Leapers": 'Tyranids',
    "Kor'sarro Khan": 'Space_Marines',
    'INVADER ATV': 'Space_Marines',
    "Vulkan He'stan": 'Space_Marines',
    'Castellum Stronghold': 'Space_Marines',
    'Wall Of Martyrs Bunker': 'Space_Marines',
    'Wall Of Martyrs Defence Line': 'Space_Marines',
    'Wall Of Martyrs Defence Emplacement': 'Space_Marines',
    'Aegis Defence Line With Weapon Emplacement': 'Space_Marines',
    "Aun'shi": 'Tau_Empire',
    "‘Iron Hand' Straken": 'Astra_Militarum',
    'CULT DEMAGOGUE': 'Genestealer_Cult',
    'BRÔKHYR IRON-MASTER': 'Leagues_of_Votann',
    
}

# Apply the character map
mask = alfa['faction'].isna()
alfa.loc[mask, 'faction'] = alfa.loc[mask, 'mini'].map(character_map)

print(f"Sin faction: {alfa['faction'].isna().sum()}")
display(alfa[alfa['faction'].isna()])
alfa

Sin faction: 0


,mini,base_mm2,cost,faction


,mini,base_mm2,cost,faction
0,Warboss,40mm,75.0,Orks
1,Warboss In Mega Armour,50mm,80.0,Orks
2,Warboss On Warbike,100 x 40mm,75.0,Orks
3,Weirdboy,40mm,65.0,Orks
4,Big Mek In Mega Armour,40mm,90.0,Orks
...,...,...,...,...
2329,Defiler,160mm,NaN,Death_Guard
2330,Thulia Ghuld,80mm,NaN,Adeptus_Mechanicus
2331,Hastarii Exterminators,32mm,NaN,Adeptus_Mechanicus
2332,Hastarii Fusiliers,32mm,NaN,Adeptus_Mechanicus


In [16]:
# Merge con v1 para precio
delta = alfa.merge(
    v1[['mini', 'price']].rename(columns={'price': 'price'}),
    on='mini',
    how='left'
)

# Merge con v2 para precio
delta = delta.merge(
    v2[['mini', 'price']].rename(columns={'price': 'price_old'}),
    on='mini',
    how='left'
)

# Precio combinado: usa v2 si existe, si no v1
delta['price'] = delta['price_old'].fillna(delta['price'])

delta[delta['price'].isna()]

,mini,base_mm2,cost,faction,price,price_old
0,Warboss,40mm,75.0,Orks,NaN,NaN
1,Warboss In Mega Armour,50mm,80.0,Orks,NaN,NaN
2,Warboss On Warbike,100 x 40mm,75.0,Orks,NaN,NaN
4,Big Mek In Mega Armour,40mm,90.0,Orks,NaN,NaN
5,Big Mek On Warbike,90 x 52mm,105.0,Orks,NaN,NaN
...,...,...,...,...,...,...
2358,Starfangs,105 x 70mm,NaN,Thousand_Sons,NaN,NaN
2363,Starfangs,105 x 70mm,NaN,Thousand_Sons,NaN,NaN
2364,Ferren Areios,40mm,NaN,Craftworlds,NaN,NaN
2373,Hastarii Exterminators,32mm,NaN,Adeptus_Mechanicus,NaN,NaN


In [17]:
from difflib import SequenceMatcher

# Miniaturas con precio conocido
known_prices = pd.concat([
    v1[['mini', 'price']].rename(columns={'price': 'p'}),
    v2[['mini', 'price']].rename(columns={'price': 'p'})
]).drop_duplicates('mini').set_index('mini')['p']

def find_price(mini_name, known, threshold=0.8):
    mini_lower = str(mini_name).lower()
    best_score = 0
    best_price = np.nan
    for known_mini, price in known.items():
        score = SequenceMatcher(None, mini_lower, str(known_mini).lower()).ratio()
        if score > best_score:
            best_score = score
            best_price = price
    return best_price if best_score >= threshold else np.nan

mask = delta['price'].isna()
delta.loc[mask, 'price'] = delta.loc[mask, 'mini'].apply(
    lambda x: find_price(x, known_prices)
)

# Rellenar restantes con mediana por faction
median_by_faction = delta.groupby('faction')['price'].median()
delta['price'] = delta.apply(
    lambda row: median_by_faction[row['faction']] if pd.isna(row['price']) else row['price'],
    axis=1
)

delta['price'] = delta['price'].fillna(delta['price'].median())

print(f"NaNs en price: {delta['price'].isna().sum()}")
delta

NaNs en price: 0


,mini,base_mm2,cost,faction,price,price_old
0,Warboss,40mm,75.0,Orks,55.20,NaN
1,Warboss In Mega Armour,50mm,80.0,Orks,40.02,NaN
2,Warboss On Warbike,100 x 40mm,75.0,Orks,55.20,NaN
3,Weirdboy,40mm,65.0,Orks,30.82,NaN
4,Big Mek In Mega Armour,40mm,90.0,Orks,67.62,NaN
...,...,...,...,...,...,...
2371,Defiler,160mm,NaN,Death_Guard,133.40,NaN
2372,Thulia Ghuld,80mm,NaN,Adeptus_Mechanicus,59.80,NaN
2373,Hastarii Exterminators,32mm,NaN,Adeptus_Mechanicus,59.80,NaN
2374,Hastarii Fusiliers,32mm,NaN,Adeptus_Mechanicus,59.80,NaN


In [18]:
from difflib import SequenceMatcher

# Miniaturas con precio conocido para price_old
known_prices_old = pd.concat([
    v1[['mini', 'price']].rename(columns={'price': 'p_old'}),
    v2[['mini', 'price']].rename(columns={'price': 'p_old'})
]).drop_duplicates('mini').set_index('mini')['p_old']

def find_price_old(mini_name, known, threshold=0.8):
    mini_lower = str(mini_name).lower()
    best_score = 0
    best_price = np.nan
    for known_mini, price in known.items():
        score = SequenceMatcher(None, mini_lower, str(known_mini).lower()).ratio()
        if score > best_score:
            best_score = score
            best_price = price
    return best_price if best_score >= threshold else np.nan

mask_old = delta['price_old'].isna()
delta.loc[mask_old, 'price_old'] = delta.loc[mask_old, 'mini'].apply(
    lambda x: find_price_old(x, known_prices_old)
)

# Rellenar restantes con mediana por faction para price_old
median_by_faction_old = delta.groupby('faction')['price_old'].median()
delta['price_old'] = delta.apply(
    lambda row: median_by_faction_old[row['faction']] if pd.isna(row['price_old']) else row['price_old'],
    axis=1
)

delta['price_old'] = delta['price_old'].fillna(delta['price_old'].median())

print(f"NaNs en price_old: {delta['price_old'].isna().sum()}")
delta

NaNs en price_old: 0


,mini,base_mm2,cost,faction,price,price_old
0,Warboss,40mm,75.0,Orks,55.20,55.20
1,Warboss In Mega Armour,50mm,80.0,Orks,40.02,40.02
2,Warboss On Warbike,100 x 40mm,75.0,Orks,55.20,55.20
3,Weirdboy,40mm,65.0,Orks,30.82,30.82
4,Big Mek In Mega Armour,40mm,90.0,Orks,67.62,67.62
...,...,...,...,...,...,...
2371,Defiler,160mm,NaN,Death_Guard,133.40,133.40
2372,Thulia Ghuld,80mm,NaN,Adeptus_Mechanicus,59.80,59.80
2373,Hastarii Exterminators,32mm,NaN,Adeptus_Mechanicus,59.80,59.80
2374,Hastarii Fusiliers,32mm,NaN,Adeptus_Mechanicus,59.80,59.80


In [19]:
# Guardar el dataframe resultante en un archivo CSV
delta.to_csv('clear_data/processed_data.csv', index=False)